# 10: Validity study on the planted-rule population (RQ1, Proposition 3, the overlap problem)

Design decisions
- Population: the 20,000 BRFSS 2015 covariate rows of notebook 04 (hash asserted). Label settings: P1, P2 and P3 at noise 0, 0.05, 0.10 and 0.20, and P4, 13 in all. All draws are with replacement from the population, with labels from the known p(x).
- Model per setting: XGBoost with 100 trees of depth 4, min_child_weight 50, gamma 2 and max_bin 32, trained on a training draw of 40,000; tau at predicted probability 0.5 (the Bayes rule for P1 to P3). Regularisation keeps the grid small enough to build candidate sets for every population cell; an unregularised model makes almost every row its own cell at several seconds per cell.
- Candidate sets for every distinct population cell with M = 20 orders (subsets M = 1, 5, 10 use the first orders); every distinct candidate box is re-checked by the second exact procedure and a sample of 50 by Z3. True box risk by Eq. (13) and exact selective risk by Eq. (14) over the population.
- Priors fixed from the training draw: pi_Occ, and pi_mix with lambda = 0.5 (training support counted over the 40,000 training patients). Naive certification (level delta_1 per box, no prior) is evaluated alongside.
- Targets fixed before the run: P1 to P3, alpha_0 = alpha_1 = alpha_sel = noise + 0.05; P4, alpha_0 = 0.05 (negative claims), alpha_1 = 0.70 (enrichment claims), alpha_sel = 0.05.
- Grid (plan E2): n_1 in {2,000; 5,000; 10,000}, M in {1, 5, 10, 20}, delta_1 in {0.01, 0.025, 0.05}; calibration split 2 of 10,000 draws with delta_2 = 0.025; R = 500 repetitions per configuration; model and candidate sets fixed, calibration draws and labels re-drawn.
- Indicators per repetition: V1, some evaluated box has true risk above its bound (checks delta_1); V2, policy certified while the true selective risk exceeds alpha_sel (checks delta_2); claim violation, some released box has true risk above its target; overlap, every released box meets its target but the true selective risk exceeds alpha_sel.
- Verdicts by the three-way rule with one-sided 95% Clopper-Pearson limits; the excess test is corrected for the number of configurations. Certification rates are reported with every result.
- Per-repetition results are stored on Drive after each setting, so an interrupted run resumes at the next setting.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scax import paths
from scax.utils import seed_everything, sha256_array, write_json
from scax.data import planted
from scax.experiments.validity import build_setting, run_repetitions
from scax.metrics.validity import validity_verdict

SEED = 42
seed_everything(SEED)
paths.ensure_dirs()
XP = pd.read_parquet(paths.DERIVED / 'planted_population.parquet')
REC04 = json.load(open(f'{paths.RESULTS}/04_population_summary.json'))
assert sha256_array(XP.to_numpy()) == REC04['population_matrix_sha256'], 'population differs from notebook 04'
P4_B0 = planted.p4_intercept(XP)
MODEL = {'n_estimators': 100, 'max_depth': 4, 'min_child_weight': 50, 'gamma': 2, 'max_bin': 32}
SETTINGS = [(r, e) for r in ('P1', 'P2', 'P3') for e in planted.NOISE_LEVELS] + [('P4', 0.0)]
N_TRAIN, M_MAX, N2, DELTA2, R = 40000, 20, 10000, 0.025, 500
N1_LIST, M_LIST, DELTA1_LIST, LAM = [2000, 5000, 10000], [1, 5, 10, 20], [0.01, 0.025, 0.05], 0.5
OUT_DIR = paths.DERIVED / 'validity'; OUT_DIR.mkdir(parents=True, exist_ok=True)


def targets(rule, eps):
    if rule == 'P4':
        return {0: 0.05, 1: 0.70}, 0.05
    a = round(eps + 0.05, 10)
    return {0: a, 1: a}, a

## Settings: model, candidate sets, verification, repetitions

In [ ]:
DIAG = []
for s_i, (rule, eps) in enumerate(SETTINGS):
    tag = f'{rule}_eps{eps:.2f}'
    reps_file, diag_file = OUT_DIR / f'{tag}_reps.parquet', OUT_DIR / f'{tag}_diag.json'
    if reps_file.exists() and diag_file.exists():
        DIAG.append(json.load(open(diag_file))); print(tag, 'loaded from Drive'); continue
    t0 = time.time()
    alpha, alpha_sel = targets(rule, eps)
    st, diag = build_setting(XP, rule, eps, P4_B0, MODEL, N_TRAIN, M_MAX, SEED + s_i, alpha, alpha_sel, lam=LAM,
                             cache_path=paths.DERIVED / 'cache' / f'10_{tag}.pkl', z3_n=50)
    t1 = time.time()
    reps = run_repetitions(st, N1_LIST, M_LIST, DELTA1_LIST, list(st.prior_logs), R, N2, DELTA2, seed=1000 + s_i)
    reps.insert(0, 'eps', eps); reps.insert(0, 'rule', rule)
    reps.to_parquet(reps_file, index=False)
    diag.update({'alpha_0': alpha[0], 'alpha_1': alpha[1], 'alpha_sel': alpha_sel, 'seconds_build': t1 - t0,
                 'seconds_repetitions': time.time() - t1})
    json.dump(diag, open(diag_file, 'w'), indent=1, default=float)
    DIAG.append(diag)
    print(f"{tag}: {diag['cells']} cells, {diag['boxes']} boxes, build {t1 - t0:.0f}s, repetitions {time.time() - t1:.0f}s")
DIAG = pd.DataFrame(DIAG)
DIAG.to_csv(f'{paths.RESULTS}/10_settings.csv', index=False)
print(DIAG[['rule', 'eps', 'cells', 'active_features', 'thresholds', 'agreement_with_bayes_rule', 'boxes',
            'unverified_removed', 'search_unknown', 'median_seconds_per_cell']].to_string(index=False))

## Aggregation and verdicts

In [ ]:
REPS = pd.concat([pd.read_parquet(OUT_DIR / f'{r}_eps{e:.2f}_reps.parquet') for r, e in SETTINGS], ignore_index=True)
keys = ['rule', 'eps', 'n1', 'delta1', 'M', 'prior']
AGG = REPS.groupby(keys).agg(reps=('V1', 'size'), V1=('V1', 'sum'), V2=('V2', 'sum'), claim_violations=('claim_violation', 'sum'),
                             overlap=('overlap', 'sum'), policy_certified=('policy_certified', 'mean'),
                             cert_rate=('cert_rate', 'mean'), R_sel=('R_sel', 'mean'), claim_precision=('claim_precision', 'mean'),
                             released_size=('released_size', 'mean'), released_support=('released_support', 'mean'),
                             feature_precision=('feature_precision', 'mean'), feature_recall=('feature_recall', 'mean'),
                             boxes_evaluated=('boxes_evaluated', 'mean')).reset_index()
K = int((AGG['prior'] != 'naive').sum())
v1 = [validity_verdict(int(r.V1), int(r.reps), float(r.delta1), k_configs=K) for r in AGG.itertuples()]
v2 = [validity_verdict(int(r.V2), int(r.reps), DELTA2, k_configs=K) for r in AGG.itertuples()]
AGG['V1_rate'] = [v['rate'] for v in v1]; AGG['V1_upper'] = [v['upper'] for v in v1]; AGG['V1_verdict'] = [v['verdict'] for v in v1]
AGG['V2_rate'] = [v['rate'] for v in v2]; AGG['V2_upper'] = [v['upper'] for v in v2]; AGG['V2_verdict'] = [v['verdict'] for v in v2]
AGG.to_csv(f'{paths.RESULTS}/10_validity.csv', index=False)
scax = AGG[AGG.prior != 'naive']
print('configurations (SCAX priors):', K)
print('V1 verdicts:', scax.V1_verdict.value_counts().to_dict())
print('V2 verdicts:', scax.V2_verdict.value_counts().to_dict())
print('naive V1 verdicts:', AGG[AGG.prior == 'naive'].V1_verdict.value_counts().to_dict())
view = AGG[(AGG.n1 == 10000) & (AGG.delta1 == 0.025) & (AGG.M == 20)]
print(view[['rule', 'eps', 'prior', 'V1_rate', 'V1_upper', 'claim_violations', 'cert_rate', 'R_sel', 'policy_certified',
            'V2_rate', 'overlap', 'released_size', 'feature_recall']].round(4).to_string(index=False))

## Figures

In [ ]:
FIG = paths.FIGURES; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
BLUE, VERMILION, GREEN = '#0072B2', '#D55E00', '#009E73'

# Figure: certificate violation frequency against the number of candidate orders M, averaged over the
# ten settings in which boxes have nonzero true risk (P1 to P3 with noise above 0, and P4)
sub = AGG[(AGG.delta1 == 0.025) & ((AGG.eps > 0) | (AGG.rule == 'P4'))]
SERIES = (('naive', VERMILION, 'o', 'Per-box bound without prior'), ('occam', BLUE, 's', 'SCAX, Occam prior'),
          (f'mix{LAM:g}', GREEN, '^', 'SCAX, mixture prior'))
fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.9), sharey=True)
for ax, n1 in zip(axes, N1_LIST):
    s = sub[sub.n1 == n1].groupby(['prior', 'M']).V1_rate.mean().unstack(0)
    for prior, col, mk, lab in SERIES:
        ax.plot(s.index, s[prior], marker=mk, color=col, lw=2, ms=5, label=lab)
    ax.axhline(0.025, color='0.4', lw=1, ls='--')
    ax.set_title(f'{n1:,} calibration records', fontsize=9)
    ax.set_xlabel('Feature orders per patient (M)')
    ax.set_xticks(M_LIST); ax.set_ylim(-0.02, 1.02)
axes[0].set_ylabel('Share of repetitions with\na box above its bound')
axes[0].text(20, 0.06, 'target 0.025', ha='right', fontsize=7, color='0.3')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.1, 1, 1))
for ext in ('png', 'pdf'):
    fig.savefig(FIG / f'fig_validity_naive_vs_scax.{ext}', dpi=600, bbox_inches='tight')
plt.show()

# Figure: the overlap problem on P4. Left: every released box meets its target, but the error rate over all
# released patients is above alpha_sel. Right: the policy is certified although that error rate is above alpha_sel.
p4 = AGG[(AGG.rule == 'P4') & (AGG.delta1 == 0.025) & (AGG.n1 == 10000)]
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), sharey=True)
for prior, col, mk, lab in SERIES:
    s = p4[p4.prior == prior].sort_values('M')
    axes[0].plot(s.M, s.overlap / s.reps, marker=mk, color=col, lw=2, ms=5, label=lab)
    axes[1].plot(s.M, s.V2 / s.reps, marker=mk, color=col, lw=2, ms=5, label=lab)
axes[0].set_title('Every released box within target,\nreleased patients above target', fontsize=9)
axes[1].set_title('Policy certified while released\npatients are above target', fontsize=9)
axes[1].axhline(DELTA2, color='0.4', lw=1, ls='--')
axes[1].text(20, DELTA2 + 0.03, 'target 0.025', ha='right', fontsize=7, color='0.3')
for ax in axes:
    ax.set_xlabel('Feature orders per patient (M)'); ax.set_xticks(M_LIST); ax.set_ylim(-0.02, 1.02)
axes[0].set_ylabel('Share of repetitions')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.1, 1, 1))
for ext in ('png', 'pdf'):
    fig.savefig(FIG / f'fig_validity_overlap_p4.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
write_json(f'{paths.RESULTS}/10_validity_summary.json', {
    'seed': SEED, 'model': MODEL, 'n_train': N_TRAIN, 'n2': N2, 'delta2': DELTA2, 'repetitions': R,
    'grid': {'n1': N1_LIST, 'M': M_LIST, 'delta1': DELTA1_LIST, 'priors': ['occam', f'mix{LAM:g}', 'naive']},
    'configurations_scax': K,
    'V1_verdicts_scax': scax.V1_verdict.value_counts().to_dict(), 'V2_verdicts_scax': scax.V2_verdict.value_counts().to_dict(),
    'V1_verdicts_naive': AGG[AGG.prior == 'naive'].V1_verdict.value_counts().to_dict(),
    'max_V1_rate_scax': float(scax.V1_rate.max()), 'max_V2_rate_scax': float(scax.V2_rate.max()),
    'settings': DIAG.to_dict('records')})
print('written: results/10_validity.csv, 10_settings.csv, 10_validity_summary.json, figures')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "10: validity study on the planted-rule population (13 label settings, 468 configurations per prior, 500 repetitions), naive comparison, overlap problem, figures"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)